# Order Book Imbalance (OBI) study

Reads `snapshots.csv` from the C++ engine. Computes top-of-book OBI, forward mid returns, and **test-set** correlations only (time-based 70/30 split).

**Not a trading strategy** — no costs, queue position, or capacity. Interpret cautiously.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path("..").resolve()
SNAP_PATH = ROOT / "data" / "snapshots_replay.csv"
TRAIN_FRAC = 0.70
HORIZONS_NS = {
    "100ms": 100_000_000,
    "1s": 1_000_000_000,
    "5s": 5_000_000_000,
}
PRIMARY = "1s"

In [ ]:
df = pd.read_csv(SNAP_PATH)
df = df.dropna(subset=["best_bid", "best_ask", "bid_sz_1", "ask_sz_1"])
df = df[(df["bid_sz_1"] > 0) & (df["ask_sz_1"] > 0)].copy()

df["mid"] = 0.5 * (df["best_bid"] + df["best_ask"])
denom = df["bid_sz_1"] + df["ask_sz_1"]
df["obi"] = (df["bid_sz_1"] - df["ask_sz_1"]) / denom

print(f"rows={len(df):,}  time span ns={df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]:,}")
df.head()

In [ ]:
def forward_mid_return(timestamps: np.ndarray, mids: np.ndarray, horizon_ns: int) -> np.ndarray:
    """For each t, find first mid at timestamp >= t + horizon (no look-ahead on features)."""
    n = len(timestamps)
    out = np.full(n, np.nan)
    j = 0
    for i in range(n):
        target = timestamps[i] + horizon_ns
        if j < i:
            j = i
        while j < n and timestamps[j] < target:
            j += 1
        if j < n and mids[i] != 0:
            out[i] = (mids[j] - mids[i]) / mids[i]
    return out

ts = df["timestamp"].to_numpy()
mid = df["mid"].to_numpy(dtype=float)
for name, h in HORIZONS_NS.items():
    df[f"ret_{name}"] = forward_mid_return(ts, mid, h)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(df["timestamp"], df["mid"], color="steelblue", lw=0.8, label="mid (ticks)")
ax1.set_ylabel("mid (ticks)")
ax2 = ax1.twinx()
ax2.plot(df["timestamp"], df["obi"], color="darkorange", lw=0.6, alpha=0.7, label="OBI")
ax2.set_ylabel("OBI")
ax1.set_title("Mid price vs top-of-book OBI")
fig.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
ret_col = f"ret_{PRIMARY}"
split = int(len(df) * TRAIN_FRAC)
train = df.iloc[:split]
test = df.iloc[split:].dropna(subset=[ret_col])

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(test["obi"], test[ret_col], s=6, alpha=0.35, c="teal")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("OBI_t")
ax.set_ylabel(f"forward mid return ({PRIMARY})")
ax.set_title(f"Test set: OBI vs {PRIMARY} return (n={len(test):,})")
plt.tight_layout()
plt.show()

In [ ]:
def report_corr(name: str, x: np.ndarray, y: np.ndarray) -> None:
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 3:
        print(f"{name}: too few points")
        return
    pearson = stats.pearsonr(x, y)
    spearman = stats.spearmanr(x, y)
    print(
        f"{name}: n={len(x):,}  "
        f"Pearson r={pearson.statistic:.4f} (p={pearson.pvalue:.2e})  "
        f"Spearman ρ={spearman.statistic:.4f} (p={spearman.pvalue:.2e})"
    )

print("=== TEST SET ONLY (last 30% by time) ===")
for name in HORIZONS_NS:
    col = f"ret_{name}"
    sub = test.dropna(subset=[col]) if name == PRIMARY else df.iloc[split:].dropna(subset=[col])
    report_corr(name, sub["obi"].to_numpy(), sub[col].to_numpy())

print("\nTrain-set correlations are for diagnostics only — do not quote as results.")
for name in HORIZONS_NS:
    col = f"ret_{name}"
    sub = train.dropna(subset=[col])
    report_corr(f"train/{name}", sub["obi"].to_numpy(), sub[col].to_numpy())

## Conclusion template

- If test-set correlation is positive and stable across horizons, OBI carries **some** short-horizon information on this synthetic/sample day.
- This does **not** imply a profitable strategy: spreads, fees, latency, and adverse selection dominate microstructure PnL.
- Synthetic data embeds a mild OBI→drift link by construction; real LOBSTER days may look weaker or regime-dependent.
- Next rigor steps (optional): multi-level OBI, longer samples, transaction-cost haircut, cross-sectional symbols.